<a href="https://colab.research.google.com/github/sharmai309/langChainLearning/blob/main/2_1_Langchain_multiple_tasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#If not installed
#!pip install transformers torch

In [2]:
#If not installed
#!pip install -U langchain-huggingface

In [3]:
#!pip install langchain-huggingface

In [4]:
import torch
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate

In [5]:
# Load the model and tokenizer locally
model_name = "google/flan-t5-base"  # You can also use "google/flan-t5-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [6]:
# Create a text generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    dtype=torch.float16,  # Uses lower precision for efficiency
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalL

In [7]:
#To control generation settings
generation_kwargs = {
    "temperature": 0.7,   # Controls randomness (higher = more creative)
    #"max_length": 512,    # Max number of tokens in response
    "max_new_tokens": 20, # max tokens
    #"min_length": 150,      # Forces at least 150 words (~150 tokens)
    "top_p": 0.9,         # Nucleus sampling (higher = more diverse responses)
    "top_k": 50,          # Limits the number of top tokens considered
    "repetition_penalty": 1.2,  # Penalizes repetition (1.0 = no penalty)
    "do_sample": True,    # Enables sampling (for creative responses)
}


In [8]:
# Wrap pipeline in LangChain's HuggingFacePipeline with parameters
llm = HuggingFacePipeline(pipeline=pipe, model_kwargs=generation_kwargs)

In [9]:
# Define prompt template
template = """Question: {question}
Answer: Let's think step by step."""
prompt = PromptTemplate(template=template, input_variables=["question"])

In [10]:
# Example questions
questions = [
    "Explain the concept of black holes in simple terms.",
    "What are the main causes of climate change, and how can we address them?",
    "Provide a brief overview of the history of artificial intelligence."
]

In [11]:
#RunnableSequence
# from langchain.schema.runnable import RunnableSequence
chain = prompt | llm

for q in questions:
     print(f"\nQ: {q}")
     print(chain.invoke({"question": q}))


Q: Explain the concept of black holes in simple terms.


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Explain the concept of black holes in simple terms.
Answer: Let's think step by step.arc in the Universe. The answer: black holes.

Q: What are the main causes of climate change, and how can we address them?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What are the main causes of climate change, and how can we address them?
Answer: Let's think step by step.not temperature, but climate change is caused by human activities, such as agriculture or fishing. The answer: climate change.

Q: Provide a brief overview of the history of artificial intelligence.


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Provide a brief overview of the history of artificial intelligence.
Answer: Let's think step by step. Intelligence was developed to develop a computer vision approach to understanding human emotions and behaviors. So the answer is artificial intelligence.


In [12]:
##Creating separate llmchain instances - using different models

In [13]:
# Define prompt
template = "Question: {question}\nAnswer: Let's think step by step."
prompt = PromptTemplate(template=template, input_variables=["question"])

In [14]:
# Load the model and tokenizer locally
model1_name = "google/flan-t5-base"  # You can also use "google/flan-t5-xl"
#model2_name = "tiiuae/falcon-7b-instruct"
model2_name = "google/flan-t5-large"


In [15]:
# Create a text generation pipeline
pipe1 = pipeline(
    "text-generation",
    model=model1_name,
    dtype=torch.float32,  # Uses lower precision for efficiency
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'Deepseek

In [16]:
#If using falcon, we could use this, note the change in task
"""
pipe2 = pipeline(
    "text-generation",
    model=model2_name,
    torch_dtype=torch.float32,  # Uses lower precision for efficiency
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)
"""

'\npipe2 = pipeline(\n    "text-generation",\n    model=model2_name,\n    torch_dtype=torch.float32,  # Uses lower precision for efficiency\n    device=0 if torch.cuda.is_available() else -1  # Use GPU if available\n)\n'

In [17]:
pipe2 = pipeline(
    "text-generation",
    model=model2_name,
    dtype=torch.float32,  # Uses lower precision for efficiency
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'Deepseek

In [18]:
llm1 = HuggingFacePipeline(pipeline=pipe1)
llm2= HuggingFacePipeline(pipeline=pipe2)

In [19]:
chain1 = prompt | llm1
chain2 = prompt | llm2

In [20]:
# Use a model based on user choice
question = "What is quantum mechanics?"
model_choice = "flan-t5"  # Example of selecting a model dynamically

In [21]:
if model_choice == "flan-t5":
    response = chain1.invoke({"question": question})
else:
    response = chain2.invoke({"question": question})

print(response)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is quantum mechanics?
Answer: Let's think step by step.articles by a physical force. Quantum mechanics is the process of determining the value of a solid by a physical force. The answer: quantum mechanics.


In [22]:
#!pip install --upgrade transformers

In [23]:
#Using a Function to Dynamically Select the Model (unc0mment below code)
#Shared Prompt Template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}")  # The input variable is "input"
])

#Function to create a HuggingFacePipeline for any model

def create_hf_llm(path):
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSeq2SeqLM.from_pretrained(path)  # or AutoModelForCausalLM

    pipe = pipeline(
        "text-generation",        # ← change from "text2text-generation"
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=256,
    )
    return HuggingFacePipeline(pipeline=pipe)


#Instantiate models
# Map model names to HuggingFace model IDs
model_map = {
    "flan-t5-base": "google/flan-t5-base",
    "flan-t5-large": "google/flan-t5-large"
}

# Wrap them as HuggingFacePipeline objects
llm_map = {name: create_hf_llm(path) for name, path in model_map.items()}

# Build LCEL-style chains: prompt | llm
chain_map = {name: prompt | llm for name, llm in llm_map.items()}


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM'

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'Deepseek

In [24]:
#Function to dynamicaly invoke the chain
def ask_question(question: str, model_choice: str):
    if model_choice not in chain_map:
        raise ValueError(f"Model {model_choice} not found. Choose from {list(chain_map.keys())}")

    chain = chain_map[model_choice]
    response = chain.invoke({"input": question})  # key must match prompt variable
    return response

In [25]:
question = "Explain relativity in simple terms."

response_base = ask_question(question, "flan-t5-base")
print("FLAN-T5-Base:", response_base)

response_large = ask_question(question, "flan-t5-large")
print("FLAN-T5-Large:", response_large)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FLAN-T5-Base: System: You are a helpful assistant.
Human: Explain relativity in simple terms.


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FLAN-T5-Large: System: You are a helpful assistant.
Human: Explain relativity in simple terms.
